# T6SE features on GPU (Google Colab)

AAC, DPC, QSO and physicochemical (CTDC / CTDT) features for the Bastion6 training sets (138 T6SE + 1112 non-T6SE).

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Edit `REPO_URL` below to point to your GitHub repository.

In [ ]:
!nvidia-smi -L

## 1. Get the code and install dependencies

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/t6se-features.git"   # <-- change me

!git clone -q {REPO_URL} t6se-features
%cd t6se-features
!pip install -q -r requirements.txt
!pip install -q cupy-cuda12x      # no-op if Colab already ships CuPy
!pip install -q -e . --no-deps

## 2. Check the backend

In [ ]:
from t6se_features import get_backend
xp, name = get_backend("auto")
print("backend:", name)
if name == "gpu":
    import cupy as cp
    print(cp.cuda.runtime.getDeviceProperties(0)["name"].decode())

## 3. Load the data (Biopython SeqIO) and compute all features

In [ ]:
import time, pandas as pd
from t6se_features import load_dataset, compute_features, write_table

df = load_dataset("data/T6SE_Training_Pos_138.fasta", "data/T6SE_Training_Neg_1112.fasta")
print(df.groupby("class").size())

GROUPS = ["AAC", "DPC", "QSO", "CTDC", "CTDT", "PYPREDT6_PHYSCHEM"]
t0 = time.perf_counter()
feats = compute_features(df.seq.tolist(), ids=df.id.tolist(), groups=GROUPS, device="auto")
print(f"done in {time.perf_counter()-t0:.2f} s")
{g: t.shape for g, t in feats.items()}

## 4. Save one table per feature (positives + negatives, `label` column: 1 = T6SE, 0 = non-T6SE)

In [ ]:
import os
os.makedirs("results", exist_ok=True)
meta = pd.DataFrame({"label": df.label.values, "class": df["class"].values,
                     "length": df.seq.str.len().values}, index=pd.Index(df.id, name="id"))
for g, t in feats.items():
    print(write_table(pd.concat([meta, t], axis=1), f"results/{g}", "csv"))
write_table(pd.concat([meta] + list(feats.values()), axis=1), "results/all_features", "csv")
pd.concat([meta, feats["CTDT"]], axis=1).head()

## 5. Sanity check: GPU result equals CPU result, and timing

The dataset is small, so the GPU is not expected to be faster here; the speed-up appears on large genome-scale inputs.

In [ ]:
import numpy as np
def run(device):
    compute_features(df.seq.tolist()[:50], groups=GROUPS, device=device)      # warm-up
    t0 = time.perf_counter()
    out = compute_features(df.seq.tolist(), ids=df.id.tolist(), groups=GROUPS, device=device)
    return out, time.perf_counter() - t0

cpu, t_cpu = run("cpu")
print(f"CPU: {t_cpu:.2f} s")
if name == "gpu":
    gpu, t_gpu = run("gpu")
    print(f"GPU: {t_gpu:.2f} s")
    for g in GROUPS:
        err = np.abs(cpu[g].values - gpu[g].values).max()
        print(f"{g:<18} max |CPU-GPU| = {err:.1e}")
        assert err < 1e-9

## 6. Run the unit tests (compare with propy3 and a naive reference)

In [ ]:
!pip install -q pytest
!pytest tests -q

## 7. Download the results

In [ ]:
!zip -qr t6se_results.zip results
from google.colab import files
files.download("t6se_results.zip")